In [4]:
"""Load the residual load in Postgres."""

import datetime as dt

from uuid import UUID
from zoneinfo import ZoneInfo

from impuls_algo.interfaces.settings import EnappsysSettings
from impuls_algo.interfaces.postgres import PostgresDbSettings, PostgresInterface
from impuls_algo.models.data import Metric, PostgresDataId, Scope
from impuls_interfaces.aws import BaseSsmParameters
from pydantic import ConfigDict, Json


class SharedSettings(BaseSsmParameters):
    """Shared Settings."""

    model_config = ConfigDict(arbitrary_types_allowed=True)

    __ssm_namespace__ = "/algo/shared"

    db: Json[PostgresDbSettings]
    enappsys: Json[EnappsysSettings]


settings = SharedSettings.from_ssm()

delivery_period_start = dt.datetime(2026, 1, 1)
delivery_period_end = dt.datetime(2026, 9, 1)


# prepare data input
daa_price_data_id: PostgresDataId = PostgresDataId(
    metric=Metric(id=UUID("a1306aea-1cf3-4f37-bcc8-77ee277741d9"), name="daa_price"),
    scope=Scope(id=UUID("89d1f2da-771c-42b5-b24a-79c9baf60c68"), name="de"),
    table="actual",
    db_schema="public",
)

# initialize postgres interface


postgres_interface = PostgresInterface(
    db_settings=settings.db,
    local_zoneinfo=ZoneInfo("Europe/Berlin"),
)

# load data
daa_results = postgres_interface.load_data(
    data_ids=[daa_price_data_id],
    delivery_period_start=delivery_period_start,
    delivery_period_end=delivery_period_end,
)

print(daa_results)

2026-09-09 11:56:44.387 | DEBUG    | impuls_interfaces.db.postgres.connection:create_iam_db_connection:93 - DB engine has already been created, create_iam_db_connection(...) has no effect
2026-09-09 11:56:44.388 | INFO     | impuls_algo.interfaces.postgres:load_data:93 - Load daa_price for de from postgres.


shape: (23_328, 2)
┌─────────────────────────────┬───────────┐
│ delivery_start              ┆ daa_price │
│ ---                         ┆ ---       │
│ datetime[μs, Europe/Berlin] ┆ f64       │
╞═════════════════════════════╪═══════════╡
│ 2026-01-01 01:00:00 CET     ┆ 60.01     │
│ 2026-01-01 01:15:00 CET     ┆ 48.9      │
│ 2026-01-01 01:30:00 CET     ┆ 36.74     │
│ 2026-01-01 01:45:00 CET     ┆ 23.93     │
│ 2026-01-01 02:00:00 CET     ┆ 36.01     │
│ …                           ┆ …         │
│ 2026-09-01 00:45:00 CEST    ┆ 147.26    │
│ 2026-09-01 01:00:00 CEST    ┆ 149.84    │
│ 2026-09-01 01:15:00 CEST    ┆ 146.05    │
│ 2026-09-01 01:30:00 CEST    ┆ 146.19    │
│ 2026-09-01 01:45:00 CEST    ┆ 144.35    │
└─────────────────────────────┴───────────┘


In [5]:
daa_results.write_parquet("daa_price.parquet")